# Phasors and AC Circuits: Impedance as a Complex Number

Alternating-current circuits are where complex numbers stopped being a mathematician's curiosity and became an everyday engineering tool. A sinusoidal voltage $V_m\cos(\omega t + \varphi)$ is the **real part of a rotating complex number** $V e^{i\omega t}$. Once you accept that, differentiation becomes multiplication by $i\omega$, resistors, inductors and capacitors all obey a single complex **Ohm's law** $V = ZI$, and solving a circuit reduces to complex arithmetic instead of differential equations.

This notebook builds the phasor method from Euler's formula, derives the impedances of $R$, $L$ and $C$, analyses series and parallel RLC circuits including **resonance**, and finally checks every phasor prediction against a direct numerical simulation of the circuit's differential equation.

This is notebook 01 of module 07 (**Applications**) in the **Learning Complex Numbers** series.

## 1. Introduction

The mains socket in your wall delivers a voltage that oscillates like a cosine, 50 or 60 times per second. Feed it into a circuit containing coils (inductors) and capacitors and the current that flows is again a cosine at the same frequency — but with a **different amplitude** and **shifted in time**. Two numbers describe the response: how much bigger or smaller, and how much earlier or later.

Two real numbers, amplitude and phase, is exactly what one complex number $re^{i\theta}$ stores. That is the whole idea of a **phasor**: replace the sinusoid by the complex number that encodes its amplitude and phase, do the algebra in $\mathbb{C}$, and convert back at the end. The engineer Charles Steinmetz popularised this in 1893, and it remains the standard method of AC circuit analysis.

> **Notation.** Electrical engineers write the imaginary unit as $j$, because $i$ (or $i(t)$) is reserved for current. Python's `1j` comes from the same tradition. In this notebook the markdown keeps the series convention $i$ for $\sqrt{-1}$, and writes currents as $I$ (phasor) and $i(t)$ (time signal) — the context always makes clear which is meant.

**Prerequisites**

- Euler's formula $e^{i\theta} = \cos\theta + i\sin\theta$ — [Euler's formula](../01_polar_form_and_euler/03_eulers_formula.ipynb).
- Multiplication as rotation and scaling — [Multiplication is rotation and scaling](../01_polar_form_and_euler/02_multiplication_is_rotation_and_scaling.ipynb).
- Division and the modulus — [Conjugate, modulus and division](../00_foundations/03_conjugate_modulus_and_division.ipynb).
- Sinusoids, amplitude and phase — [Trigonometric functions](../../../LearningCalculus/hypersklill/00_basics/07_trigonometric_functions.ipynb) (LearningCalculus).

After completing this notebook you will be able to:

- Convert between a sinusoid $V_m\cos(\omega t + \varphi)$ and its phasor $V = V_m e^{i\varphi}$.
- Explain why $\frac{d}{dt}$ becomes multiplication by $i\omega$, and derive the impedances $R$, $i\omega L$ and $\frac{1}{i\omega C}$.
- Combine impedances in series and parallel, and compute currents, voltages and phase shifts.
- Find the resonance frequency, bandwidth and quality factor of a series RLC circuit.
- Verify a phasor answer by simulating the circuit's differential equation numerically.

## 2. Theory

### 2.1 A sinusoid is the shadow of a rotating vector

Every sinusoid of angular frequency $\omega$ can be written as

$$
v(t) = V_m \cos(\omega t + \varphi) = \mathrm{Re}\left(V_m e^{i(\omega t + \varphi)}\right) = \mathrm{Re}\left(V e^{i\omega t}\right), \qquad V = V_m e^{i\varphi}.
$$

The complex number $V$ is the **phasor** of $v(t)$: its modulus $|V| = V_m$ is the amplitude and its argument $\arg V = \varphi$ is the phase. Geometrically, $V e^{i\omega t}$ is the arrow $V$ spinning counter-clockwise at $\omega$ radians per second, and $v(t)$ is its **shadow on the real axis**. Freezing the picture at $t = 0$ leaves just $V$.

A sine is a cosine delayed by a quarter turn, $\sin(\omega t) = \cos(\omega t - \tfrac{\pi}{2})$, so $V_m\sin(\omega t + \varphi)$ has phasor $V_m e^{i(\varphi - \pi/2)} = -i\,V_m e^{i\varphi}$.

### 2.2 Linear operations act on phasors

Because $\mathrm{Re}$ is linear over real scalars and all signals share the same factor $e^{i\omega t}$:

- **Addition.** $\mathrm{Re}(V_1 e^{i\omega t}) + \mathrm{Re}(V_2 e^{i\omega t}) = \mathrm{Re}\left((V_1 + V_2)e^{i\omega t}\right)$. Adding sinusoids of the same frequency is adding arrows.
- **Differentiation.** $\frac{d}{dt}\,\mathrm{Re}(V e^{i\omega t}) = \mathrm{Re}(i\omega V e^{i\omega t})$. Differentiating multiplies the phasor by $i\omega$: scale by $\omega$ and **rotate by $+90°$** (the derivative of a cosine leads it by a quarter period).
- **Integration** (of the oscillating part) divides the phasor by $i\omega$: scale by $1/\omega$ and rotate by $-90°$.

The phasor is unique: if $\mathrm{Re}(V e^{i\omega t}) = \mathrm{Re}(W e^{i\omega t})$ for all $t$, then $V = W$ (take $t = 0$ and $t = \frac{\pi}{2\omega}$ to match real and imaginary parts).

### 2.3 Impedance: Ohm's law for every component

Write the voltage across a component and the current through it as $v(t) = \mathrm{Re}(Ve^{i\omega t})$ and $i(t) = \mathrm{Re}(Ie^{i\omega t})$.

| Component | Time-domain law | Phasor law | Impedance $Z = V/I$ |
|---|---|---|---|
| Resistor $R$ (ohms) | $v = R\,i$ | $V = R\,I$ | $Z_R = R$ |
| Inductor $L$ (henries) | $v = L\,\dfrac{di}{dt}$ | $V = i\omega L\,I$ | $Z_L = i\omega L$ |
| Capacitor $C$ (farads) | $i = C\,\dfrac{dv}{dt}$ | $I = i\omega C\,V$ | $Z_C = \dfrac{1}{i\omega C} = -\dfrac{i}{\omega C}$ |

So $V = ZI$ for all three. The **impedance** $Z = R_Z + iX$ is a complex number measured in ohms: its real part is the **resistance**, its imaginary part the **reactance** $X$. In polar form $Z = |Z|e^{i\theta}$, the law $V = ZI$ says

$$
|V| = |Z|\,|I|, \qquad \arg V = \arg Z + \arg I.
$$

The modulus scales the amplitude and the argument is the **phase lead of the voltage over the current**. An inductor ($\theta = +90°$) makes the current lag the voltage by a quarter period; a capacitor ($\theta = -90°$) makes it lead.

The reciprocal $Y = 1/Z$ is the **admittance** (measured in siemens).

### 2.4 Series and parallel

Kirchhoff's laws are linear, so they hold for phasors exactly as for direct currents:

$$
Z_{\text{series}} = Z_1 + Z_2 + \cdots, \qquad \frac{1}{Z_{\text{parallel}}} = \frac{1}{Z_1} + \frac{1}{Z_2} + \cdots
$$

### 2.5 The series RLC circuit and resonance

A resistor, inductor and capacitor in series, driven by $v(t) = V_m\cos\omega t$, has

$$
Z(\omega) = R + i\left(\omega L - \frac{1}{\omega C}\right), \qquad I = \frac{V}{Z(\omega)}.
$$

As $\omega$ runs from $0$ to $\infty$, the reactance runs from $-\infty$ to $+\infty$ and $Z(\omega)$ moves **up the vertical line** $\mathrm{Re}\,Z = R$ in the complex plane. Its modulus is smallest when the reactance vanishes:

$$
\omega_0 L = \frac{1}{\omega_0 C} \quad\Longleftrightarrow\quad \omega_0 = \frac{1}{\sqrt{LC}}.
$$

At this **resonance frequency** the inductor and capacitor voltages cancel exactly, $Z = R$ is real, the current is in phase with the voltage and has its maximum amplitude $V_m/R$. Two numbers describe how sharp the peak is:

- **Quality factor** $Q = \dfrac{\omega_0 L}{R} = \dfrac{1}{R}\sqrt{\dfrac{L}{C}}$.
- **Bandwidth** $\Delta\omega = \omega_2 - \omega_1 = \dfrac{R}{L} = \dfrac{\omega_0}{Q}$, where $\omega_{1,2}$ are the half-power frequencies at which $|X| = R$, so $|Z| = \sqrt{2}R$ and $|I| = \frac{V_m}{\sqrt{2}R}$.

Below resonance the circuit is **capacitive** ($X < 0$, current leads); above it is **inductive** ($X > 0$, current lags).

### 2.6 Average power

The time-average of $v(t)i(t)$ over a period is

$$
P = \tfrac{1}{2}\,\mathrm{Re}\left(V\overline{I}\right) = \tfrac{1}{2}|V||I|\cos\theta,
$$

where $\theta = \arg V - \arg I$. Only the in-phase part of the current delivers energy; $\cos\theta$ is the **power factor**. Pure inductors and capacitors ($\theta = \pm 90°$) store and return energy but consume none on average.

## 3. Worked Examples

### Example 1 — sinusoid to phasor and back

$v(t) = 10\cos(100t + \pi/6)$ volts has phasor

$$
V = 10e^{i\pi/6} = 10\left(\tfrac{\sqrt3}{2} + \tfrac12 i\right) = 8.660 + 5i.
$$

Conversely, the phasor $I = 2 - 2i = 2\sqrt2\,e^{-i\pi/4}$ at $\omega = 100$ means $i(t) = 2\sqrt2\cos(100t - \pi/4)$.

### Example 2 — adding two sinusoids by adding arrows

$3\cos\omega t + 4\sin\omega t$: the phasors are $3$ and $-4i$ (a sine is a cosine rotated by $-90°$). Sum:

$$
3 - 4i = 5e^{-i\arctan(4/3)} = 5e^{-0.9273 i} \quad\Longrightarrow\quad 3\cos\omega t + 4\sin\omega t = 5\cos(\omega t - 0.9273).
$$

No trigonometric identities needed — the 3-4-5 triangle is visible in the complex plane.

### Example 3 — a series RL circuit

$R = 3$ ohms, $L = 0.04$ henries, $\omega = 100$ rad/s, source $v(t) = 10\cos(100t)$ volts.

1. $Z_L = i\omega L = 4i$, so $Z = 3 + 4i = 5e^{0.9273 i}$ ohms.
2. $I = \dfrac{V}{Z} = \dfrac{10}{3 + 4i} = \dfrac{10(3 - 4i)}{25} = 1.2 - 1.6i = 2e^{-0.9273 i}$ amperes.
3. Back to time: $i(t) = 2\cos(100t - 0.9273)$. The current **lags** the voltage by $0.9273$ rad $= 53.13°$, i.e. by $0.9273/100 = 9.27$ ms.

### Example 4 — series RLC below resonance

$R = 10$ ohms, $L = 0.1$ henries, $C = 100\ \mu\mathrm{F}$, source amplitude $10$ volts at $\omega = 200$ rad/s.

1. $Z_L = i \cdot 200 \cdot 0.1 = 20i$ and $Z_C = \dfrac{1}{i \cdot 200 \cdot 10^{-4}} = \dfrac{1}{0.02i} = -50i$.
2. $Z = 10 + 20i - 50i = 10 - 30i$, $|Z| = \sqrt{1000} = 31.62$ ohms, $\arg Z = -1.249$ rad.
3. $I = \dfrac{10}{10 - 30i} = \dfrac{10(10 + 30i)}{1000} = 0.1 + 0.3i$, amplitude $0.3162$ A, phase $+1.249$ rad $= 71.6°$. The circuit is capacitive, so the current **leads**.
4. Component voltages: $V_R = RI = 1 + 3i$, $V_L = 20i \cdot I = -6 + 2i$, $V_C = -50i \cdot I = 15 - 5i$. Check: $V_R + V_L + V_C = 10$. ✓ Note that $|V_C| = 15.8$ volts exceeds the source amplitude — possible because $V_L$ and $V_C$ point in opposite directions and partly cancel.
5. Resonance: $\omega_0 = 1/\sqrt{0.1 \cdot 10^{-4}} = 316.2$ rad/s, $Q = \omega_0 L / R = 3.162$, bandwidth $R/L = 100$ rad/s.

### Example 5 — a parallel combination

A resistor of $10$ ohms in parallel with a capacitor whose impedance at the operating frequency is $-10i$ ohms:

$$
Z = \frac{Z_1 Z_2}{Z_1 + Z_2} = \frac{10 \cdot (-10i)}{10 - 10i} = \frac{-100i\,(10 + 10i)}{200} = 5 - 5i \ \text{ohms}.
$$

The modulus $5\sqrt2 \approx 7.07$ is smaller than either branch alone, as for resistors in parallel.

## 4. Python Examples

### 4.1 Phasor helpers

Python's `1j` is the engineer's $j$. A phasor is just a `complex`; `cmath.rect` and `cmath.polar` convert between amplitude/phase and Cartesian form.

In [ ]:
import cmath
import math


def phasor(amplitude, phase):
    # Phasor of amplitude * cos(omega t + phase)
    return cmath.rect(amplitude, phase)


def describe(name, value, unit):
    amplitude, phase = cmath.polar(value)
    print(f"{name} = {value.real:+.4f} {value.imag:+.4f}j  ->  amplitude {amplitude:.4f} {unit}, "
          f"phase {phase:+.4f} rad ({math.degrees(phase):+.2f} deg)")


describe("V (Ex 1)", phasor(10, math.pi / 6), "volts")
describe("3 - 4j (Ex 2)", 3 - 4j, "")

omega = 100
Z = 3 + 1j * omega * 0.04
I = 10 / Z
describe("Z (Ex 3)", Z, "ohms")
describe("I (Ex 3)", I, "amperes")
print(f"current lags by {math.degrees(cmath.phase(Z)):.2f} deg = {1000 * cmath.phase(Z) / omega:.2f} ms")

### 4.2 Impedance functions and the series RLC example

In [ ]:
import cmath
import math


def impedance_resistor(R, omega):
    return complex(R, 0)


def impedance_inductor(L, omega):
    return 1j * omega * L


def impedance_capacitor(C, omega):
    return 1 / (1j * omega * C)


def series(*impedances):
    return sum(impedances)


def parallel(*impedances):
    return 1 / sum(1 / Z for Z in impedances)


R, L, C, omega = 10.0, 0.1, 1e-4, 200.0
Z_R = impedance_resistor(R, omega)
Z_L = impedance_inductor(L, omega)
Z_C = impedance_capacitor(C, omega)
Z = series(Z_R, Z_L, Z_C)
V = 10.0
I = V / Z

print("Z_L =", Z_L, "  Z_C =", Z_C)
print("Z   =", Z, "  |Z| =", round(abs(Z), 4))
print("I   =", I, "  |I| =", round(abs(I), 4), "  phase (deg) =", round(math.degrees(cmath.phase(I)), 2))

V_R, V_L, V_C = Z_R * I, Z_L * I, Z_C * I
print("V_R, V_L, V_C =", V_R, V_L, V_C)
print("KVL holds (sum equals source):", abs(V_R + V_L + V_C - V) < 1e-12)

omega_0 = 1 / math.sqrt(L * C)
print(f"omega_0 = {omega_0:.2f} rad/s, Q = {omega_0 * L / R:.3f}, bandwidth = {R / L:.1f} rad/s")
print("Ex 5 parallel:", parallel(10, -10j))

### 4.3 Resonance, symbolically

Sympy confirms that the reactance vanishes at $\omega_0 = 1/\sqrt{LC}$ and that the half-power frequencies are separated by exactly $R/L$.

In [ ]:
import sympy as sp

omega, R, L, C = sp.symbols("omega R L C", positive=True)
Z = R + sp.I * (omega * L - 1 / (omega * C))

reactance = sp.im(Z)
print("Im Z =", sp.simplify(reactance))
print("Im Z = 0 at omega =", sp.solve(sp.Eq(reactance, 0), omega))

# Half-power points: reactance = -R (below omega_0) and reactance = +R (above omega_0).
# Each is a quadratic in omega; keep the positive root.
omega_1 = (-R + sp.sqrt(R**2 + 4 * L / C)) / (2 * L)
omega_2 = (R + sp.sqrt(R**2 + 4 * L / C)) / (2 * L)
print("X(omega_1) + R =", sp.simplify(reactance.subs(omega, omega_1) + R))
print("X(omega_2) - R =", sp.simplify(reactance.subs(omega, omega_2) - R))
print("omega_2 - omega_1 =", sp.simplify(omega_2 - omega_1))
print("omega_1 * omega_2 =", sp.simplify(omega_1 * omega_2), "  (= omega_0^2)")

### 4.4 Checking the phasor answer against the differential equation

The phasor method claims to give the **steady-state** current without solving any differential equation. Let us test that claim. With $q(t)$ the capacitor charge and $i = \dot q$, Kirchhoff's voltage law for the series RLC circuit reads

$$
L\,\frac{di}{dt} + R\,i + \frac{q}{C} = V_m\cos\omega t .
$$

We integrate this numerically with a hand-written fourth-order Runge–Kutta step, starting from rest. Initially there is a transient (decaying like $e^{-Rt/(2L)} = e^{-50t}$), after which $i(t)$ should match $\mathrm{Re}(Ie^{i\omega t})$ with $I = 0.1 + 0.3i$.

In [ ]:
import numpy as np

R, L, C = 10.0, 0.1, 1e-4
V_m, omega = 10.0, 200.0


def rlc_rhs(t, state):
    charge, current = state
    source = V_m * np.cos(omega * t)
    return np.array([current, (source - R * current - charge / C) / L])


def rk4(rhs, state0, t_end, dt):
    steps = int(round(t_end / dt))
    times = np.linspace(0.0, steps * dt, steps + 1)
    states = np.zeros((steps + 1, len(state0)))
    states[0] = state0
    for k in range(steps):
        t, y = times[k], states[k]
        k1 = rhs(t, y)
        k2 = rhs(t + dt / 2, y + dt / 2 * k1)
        k3 = rhs(t + dt / 2, y + dt / 2 * k2)
        k4 = rhs(t + dt, y + dt * k3)
        states[k + 1] = y + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    return times, states


times, states = rk4(rlc_rhs, np.array([0.0, 0.0]), t_end=0.4, dt=1e-4)
simulated_current = states[:, 1]

I_phasor = V_m / (R + 1j * omega * L + 1 / (1j * omega * C))
phasor_current = np.real(I_phasor * np.exp(1j * omega * times))

late = times > 0.25
print("phasor I =", np.round(I_phasor, 6))
print("max |simulated - phasor| for t > 0.25 s:", f"{np.max(np.abs(simulated_current[late] - phasor_current[late])):.2e}")
print("max |simulated - phasor| for t < 0.05 s:", f"{np.max(np.abs(simulated_current[times < 0.05] - phasor_current[times < 0.05])):.2e}  (transient)")

After the transient dies away, simulation and phasor agree to about $10^{-6}$ A: the complex algebra of section 3 *is* the steady-state solution of the ODE. (Notebook 02 explains the transient: it is the homogeneous solution $e^{\lambda t}$ with complex $\lambda$.)

## 5. Visualizations

### 5.1 The rotating phasor and its shadow

Left: the phasor $V e^{i\omega t}$ of Example 1 at several instants — the arrow spins counter-clockwise. Right: its real part, the actual voltage. Each coloured dot on the right is the horizontal coordinate of the matching arrow tip on the left.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

V = 10 * np.exp(1j * np.pi / 6)
omega = 100.0
period = 2 * np.pi / omega
snapshot_times = np.linspace(0, period, 6, endpoint=False)
colors = plt.cm.viridis(np.linspace(0, 0.9, len(snapshot_times)))

fig, (ax_plane, ax_time) = plt.subplots(1, 2, figsize=(13, 5.5), gridspec_kw={"width_ratios": [1, 1.4]})
circle = 10 * np.exp(1j * np.linspace(0, 2 * np.pi, 300))
ax_plane.plot(circle.real, circle.imag, color="lightgray")
for t, color in zip(snapshot_times, colors):
    tip = V * np.exp(1j * omega * t)
    ax_plane.annotate("", xy=(tip.real, tip.imag), xytext=(0, 0),
                      arrowprops=dict(arrowstyle="->", color=color, linewidth=2))
    ax_plane.plot([tip.real, tip.real], [tip.imag, 0], ":", color=color)
    ax_plane.plot(tip.real, 0, "o", color=color)
ax_plane.axhline(0, color="black", linewidth=0.8)
ax_plane.axvline(0, color="black", linewidth=0.8)
ax_plane.set_xlim(-12, 12)
ax_plane.set_ylim(-12, 12)
ax_plane.set_aspect("equal")
ax_plane.set_xlabel("Re")
ax_plane.set_ylabel("Im")
ax_plane.set_title("Phasor $V e^{i\\omega t}$ at six instants")
ax_plane.grid(True, alpha=0.3)

t = np.linspace(0, 1.5 * period, 400)
ax_time.plot(t * 1000, np.real(V * np.exp(1j * omega * t)), color="tab:blue", label="$v(t) = 10\\cos(100t + \\pi/6)$")
for t_k, color in zip(snapshot_times, colors):
    ax_time.plot(t_k * 1000, np.real(V * np.exp(1j * omega * t_k)), "o", color=color, markersize=8)
ax_time.axhline(0, color="black", linewidth=0.8)
ax_time.set_xlabel("t (ms)")
ax_time.set_ylabel("volts")
ax_time.set_title("Its shadow on the real axis is the voltage")
ax_time.grid(True, alpha=0.3)
ax_time.legend(loc="upper right")
plt.tight_layout()
plt.show()

### 5.2 Phasor diagram of the series RLC circuit (Example 4)

Kirchhoff's voltage law $V_R + V_L + V_C = V$ is **vector addition**. Drawn head to tail, the three component voltages close up on the source phasor $V = 10$. $V_R$ is parallel to the current $I$; $V_L$ is $I$ rotated by $+90°$ and $V_C$ is $I$ rotated by $-90°$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

R, L, C, omega, V = 10.0, 0.1, 1e-4, 200.0, 10.0
Z_L, Z_C = 1j * omega * L, 1 / (1j * omega * C)
I = V / (R + Z_L + Z_C)
pieces = [("$V_R$", R * I, "tab:green"), ("$V_L$", Z_L * I, "tab:orange"), ("$V_C$", Z_C * I, "tab:purple")]

fig, ax = plt.subplots(figsize=(7.5, 7))
start = 0j
for label, piece, color in pieces:
    end = start + piece
    ax.annotate("", xy=(end.real, end.imag), xytext=(start.real, start.imag),
                arrowprops=dict(arrowstyle="->", color=color, linewidth=2.5))
    middle = start + piece / 2
    ax.text(middle.real + 0.4, middle.imag + 0.4, label, color=color, fontsize=13)
    start = end
ax.annotate("", xy=(V, 0), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="black", linewidth=2.5))
ax.text(V / 2, -0.9, "source $V = 10$", ha="center", fontsize=12)
scaled_current = 20 * I
ax.annotate("", xy=(scaled_current.real, scaled_current.imag), xytext=(0, 0),
            arrowprops=dict(arrowstyle="->", color="tab:red", linewidth=1.5, linestyle="--"))
ax.text(scaled_current.real + 0.3, scaled_current.imag, "$20\\,I$ (direction of current)", color="tab:red")

ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlim(-7, 13)
ax.set_ylim(-2, 8)
ax.set_aspect("equal")
ax.set_xlabel("Re (volts)")
ax.set_ylabel("Im (volts)")
ax.set_title("$V_R + V_L + V_C = V$ at $\\omega = 200$ rad/s")
ax.grid(True, alpha=0.3)
plt.show()

### 5.3 Resonance curves

Amplitude $|I(\omega)|$ and phase $\arg I(\omega)$ of the series RLC current for three resistances (same $L$, $C$, source amplitude 10 volts). Smaller $R$ means a taller, narrower peak (higher $Q$). The phase always passes through $0$ at $\omega_0 = 316$ rad/s: leading (capacitive) below, lagging (inductive) above.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

L, C, V = 0.1, 1e-4, 10.0
omega = np.linspace(50, 1000, 2000)
omega_0 = 1 / np.sqrt(L * C)

fig, (ax_amp, ax_phase) = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
for R in [5.0, 10.0, 20.0]:
    Z = R + 1j * (omega * L - 1 / (omega * C))
    I = V / Z
    Q = omega_0 * L / R
    ax_amp.plot(omega, np.abs(I), label=f"R = {R:g} ohms (Q = {Q:.2f})")
    ax_phase.plot(omega, np.degrees(np.angle(I)))

for ax in (ax_amp, ax_phase):
    ax.axvline(omega_0, color="gray", linestyle="--", linewidth=1)
    ax.grid(True, alpha=0.3)
ax_amp.set_ylabel("$|I|$ (amperes)")
ax_amp.set_title("Series RLC: current amplitude and phase")
ax_amp.legend()
ax_phase.axhline(0, color="black", linewidth=0.8)
ax_phase.set_ylabel("$\\arg I$ (degrees)")
ax_phase.set_xlabel("$\\omega$ (rad/s)")
ax_phase.text(omega_0 + 10, 60, "$\\omega_0$", color="gray")
plt.tight_layout()
plt.show()

### 5.4 Impedance and admittance loci in the complex plane

As $\omega$ sweeps, $Z(\omega) = R + iX(\omega)$ climbs the vertical line $\mathrm{Re}\,Z = R$. Its reciprocal, the admittance $Y = 1/Z = I/V$, traces a **circle** through the origin with centre $\frac{1}{2R}$ and radius $\frac{1}{2R}$ — inversion maps lines to circles (notebook 03/06 on Möbius transformations). The point of the circle farthest from the origin is resonance. The current phasor for a 1-volt source is exactly $Y$, so this circle *is* the resonance curve, seen in the complex plane. (Labels give $\omega$ in rad/s.)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

R, L, C = 10.0, 0.1, 1e-4
omega = np.geomspace(20, 5000, 2000)
Z = R + 1j * (omega * L - 1 / (omega * C))
Y = 1 / Z
marked = [150.0, 200.0, 316.23, 500.0, 700.0]
center = 1 / (2 * R)

fig, (ax_z, ax_y) = plt.subplots(1, 2, figsize=(12, 6))
ax_z.plot(Z.real, Z.imag, color="tab:blue")
ax_y.plot(Y.real, Y.imag, color="tab:red")
for w in marked:
    z_point = R + 1j * (w * L - 1 / (w * C))
    y_point = 1 / z_point
    ax_z.plot(z_point.real, z_point.imag, "ko")
    ax_z.text(z_point.real + 3, z_point.imag, f"$\\omega$ = {w:g}", fontsize=9, va="center")
    ax_y.plot(y_point.real, y_point.imag, "ko")
    label_position = center + 1.25 * (y_point - center)
    ax_y.text(label_position.real, label_position.imag, f"{w:g}", fontsize=9, ha="center", va="center")

ax_z.set_xlim(-50, 70)
ax_z.set_ylim(-65, 65)
ax_z.set_title("Impedance $Z(\\omega)$: the line Re $Z = R$")
ax_y.set_xlim(-0.04, 0.14)
ax_y.set_ylim(-0.07, 0.07)
ax_y.set_title("Admittance $Y = 1/Z$: a circle of diameter $1/R$")
for ax in (ax_z, ax_y):
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_aspect("equal")
    ax.set_xlabel("Re")
    ax.set_ylabel("Im")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.5 Time domain: transient, then the phasor prediction

The simulated current (from section 4.4) starts at zero, wobbles through a decaying transient, and then locks onto the phasor prediction $\mathrm{Re}(Ie^{i\omega t})$. The source voltage (scaled down by 20 to share the axis) peaks *after* the current: the current leads by $71.6°$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

R, L, C = 10.0, 0.1, 1e-4
V_m, omega = 10.0, 200.0
dt, t_end = 1e-4, 0.25


def rhs(t, state):
    charge, current = state
    return np.array([current, (V_m * np.cos(omega * t) - R * current - charge / C) / L])


times = np.arange(0, t_end + dt / 2, dt)
states = np.zeros((len(times), 2))
for k in range(len(times) - 1):
    t, y = times[k], states[k]
    k1 = rhs(t, y)
    k2 = rhs(t + dt / 2, y + dt / 2 * k1)
    k3 = rhs(t + dt / 2, y + dt / 2 * k2)
    k4 = rhs(t + dt, y + dt * k3)
    states[k + 1] = y + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)

I_phasor = V_m / (R + 1j * omega * L + 1 / (1j * omega * C))

plt.figure(figsize=(10, 4.5))
plt.plot(times * 1000, states[:, 1], color="tab:blue", linewidth=2, label="simulated $i(t)$ (RK4)")
plt.plot(times * 1000, np.real(I_phasor * np.exp(1j * omega * times)), "--", color="tab:orange",
         label="phasor prediction $\\mathrm{Re}(I e^{i\\omega t})$")
plt.plot(times * 1000, V_m * np.cos(omega * times) / 20, color="gray", alpha=0.6, label="source $v(t)/20$")
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("t (ms)")
plt.ylabel("amperes")
plt.title("Series RLC driven at 200 rad/s: the transient decays, the phasor solution remains")
plt.ylim(-0.55, 0.8)
plt.grid(True, alpha=0.3)
plt.legend(loc="upper right", ncol=3, fontsize=9)
plt.show()

## 6. Exercises

### Basic

1. Write the phasors of $v_1(t) = 5\cos(50t - \pi/3)$ and $v_2(t) = 2\sin(50t)$ in polar and Cartesian form.
2. A phasor current $I = -3 + 3i$ amperes at $\omega = 60$ rad/s. Write $i(t)$ as a cosine with a phase.
3. Compute the impedance of a $0.5$ henry inductor and of a $20\ \mu\mathrm{F}$ capacitor at $\omega = 100$ rad/s. Which one has the larger modulus?

### Intermediate

4. **Draw it first, then compute.** A series RC circuit has $R = 4$ ohms and $Z_C = -3i$ ohms. Sketch $Z$ in the complex plane, and from the sketch predict whether the current leads or lags the source and by roughly how much. Then compute $I$ for a source $V = 10$ volts and the exact phase.
5. Use phasors to write $\cos(\omega t) + \cos(\omega t + 2\pi/3) + \cos(\omega t + 4\pi/3)$ as a single sinusoid. (This is why balanced three-phase power needs no return wire.)
6. A series RLC circuit has $L = 20$ mH and must resonate at $\omega_0 = 5000$ rad/s with bandwidth $250$ rad/s. Find $C$, $R$ and $Q$.

### Advanced

7. For the parallel RLC circuit (all three components in parallel across the source), show that the admittance is $Y = \frac1R + i\left(\omega C - \frac{1}{\omega L}\right)$, find the frequency at which $|Z|$ is *largest*, and give its value there.
8. Prove that the average power is $P = \frac12 \mathrm{Re}(V\overline{I})$: expand $v(t)i(t)$ using $\mathrm{Re}(w) = \frac{w + \overline w}{2}$ and average over one period. Compute $P$ for Example 4 and check that it equals the power $\frac12 R|I|^2$ dissipated in the resistor.
9. Prove the claim of section 5.4: as $X$ ranges over $\mathbb{R}$, $Y = \frac{1}{R + iX}$ lies on the circle $\left|Y - \frac{1}{2R}\right| = \frac{1}{2R}$.

## 7. Solutions

1. $V_1 = 5e^{-i\pi/3} = 2.5 - 4.330i$. $v_2 = 2\cos(50t - \pi/2)$, so $V_2 = 2e^{-i\pi/2} = -2i$.
2. $|I| = 3\sqrt2 \approx 4.243$ and $\arg I = 3\pi/4$ (second quadrant), so $i(t) = 3\sqrt2\cos(60t + 3\pi/4)$.
3. $Z_L = i \cdot 100 \cdot 0.5 = 50i$ ohms. $Z_C = \frac{1}{i \cdot 100 \cdot 2\times 10^{-5}} = \frac{1}{0.002 i} = -500i$ ohms. The capacitor's modulus ($500$) is ten times larger: at low frequency a capacitor nearly blocks current.
4. Sketch: $Z = 4 - 3i$ points into the fourth quadrant, closer to the real axis than the diagonal (the real part is larger), so roughly $35°$–$40°$ below the axis. Since $I = V/Z$ has $\arg I = -\arg Z$, the current **leads** by about $37°$. Computation: $I = \frac{10}{4 - 3i} = \frac{10(4 + 3i)}{25} = 1.6 + 1.2i = 2e^{0.6435i}$; lead $= 0.6435$ rad $= 36.87°$.
5. The phasors $1, e^{2\pi i/3}, e^{4\pi i/3}$ are the three cube roots of unity, whose sum is $0$ (notebook 01/05). So the sum is identically $0$.
6. $C = \frac{1}{\omega_0^2 L} = \frac{1}{25\times10^6 \cdot 0.02} = 2\ \mu\mathrm{F}$. Bandwidth $R/L = 250$ gives $R = 250 \cdot 0.02 = 5$ ohms. $Q = \omega_0/\Delta\omega = 5000/250 = 20$.
7. Admittances add in parallel: $Y = \frac1R + \frac{1}{i\omega L} + i\omega C = \frac1R + i\left(\omega C - \frac{1}{\omega L}\right)$. $|Z| = 1/|Y|$ is largest when $|Y|$ is smallest, i.e. when the imaginary part vanishes: $\omega_0 = 1/\sqrt{LC}$ again, where $Z = R$. (The parallel circuit is an *anti*-resonator: at $\omega_0$ it draws the least current.)
8. With $v = \frac12(Ve^{i\omega t} + \overline V e^{-i\omega t})$ and similarly for $i$, the product is $\frac14\left(VIe^{2i\omega t} + \overline V\,\overline I e^{-2i\omega t} + V\overline I + \overline V I\right)$. The first two terms average to $0$ over a period; the last two give $\frac14(V\overline I + \overline{V\overline I}) = \frac12\mathrm{Re}(V\overline I)$. For Example 4: $V\overline I = 10(0.1 - 0.3i)$, so $P = \frac12 \cdot 1 = 0.5$ watts, and $\frac12 R|I|^2 = \frac12 \cdot 10 \cdot 0.1 = 0.5$ watts. ✓
9. $Y - \frac{1}{2R} = \frac{2R - (R + iX)}{2R(R + iX)} = \frac{R - iX}{2R(R + iX)}$. Numerator and denominator factor $R \mp iX$ are conjugates and have equal moduli, so $\left|Y - \frac{1}{2R}\right| = \frac{1}{2R}$.

Numerical checks:

In [ ]:
import cmath
import math

import numpy as np

print("Ex 1:", cmath.rect(5, -math.pi / 3), cmath.rect(2, -math.pi / 2))
print("Ex 2:", cmath.polar(-3 + 3j), " (3*pi/4 =", 3 * math.pi / 4, ")")
print("Ex 3:", 1j * 100 * 0.5, 1 / (1j * 100 * 2e-5))

I4 = 10 / (4 - 3j)
print("Ex 4:", I4, cmath.polar(I4), math.degrees(cmath.phase(I4)))

print("Ex 5:", abs(sum(cmath.exp(2j * math.pi * k / 3) for k in range(3))) < 1e-12)

L6, omega_0, bandwidth = 0.02, 5000.0, 250.0
print("Ex 6: C =", 1 / (omega_0**2 * L6), " R =", bandwidth * L6, " Q =", omega_0 / bandwidth)

R7, L7, C7 = 10.0, 0.1, 1e-4
omegas = np.linspace(100, 1000, 200001)
Z_parallel = 1 / (1 / R7 + 1 / (1j * omegas * L7) + 1j * omegas * C7)
print("Ex 7: argmax |Z| at omega =", round(omegas[np.argmax(np.abs(Z_parallel))], 2), " vs 1/sqrt(LC) =", round(1 / math.sqrt(L7 * C7), 2))

I_ex4 = 10 / (10 + 20j - 50j)
t = np.linspace(0, 2 * np.pi / 200, 100001)
instantaneous = 10 * np.cos(200 * t) * np.real(I_ex4 * np.exp(200j * t))
print("Ex 8: 0.5 Re(V conj I) =", 0.5 * (10 * np.conj(I_ex4)).real, " numeric average =", round(instantaneous[:-1].mean(), 6),
      " 0.5 R|I|^2 =", 0.5 * 10 * abs(I_ex4) ** 2)

X = np.linspace(-1000, 1000, 10001)
Y = 1 / (R7 + 1j * X)
print("Ex 9: all on the circle:", np.allclose(np.abs(Y - 1 / (2 * R7)), 1 / (2 * R7)))

## 8. Conclusion

- A sinusoid $V_m\cos(\omega t + \varphi)$ is the real part of the rotating vector $Ve^{i\omega t}$; its **phasor** $V = V_m e^{i\varphi}$ stores amplitude as modulus and phase as argument.
- Adding same-frequency sinusoids is adding phasors; differentiating multiplies by $i\omega$ (a $90°$ rotation).
- Every passive component obeys $V = ZI$ with **impedance** $R$, $i\omega L$ or $\frac{1}{i\omega C}$; series impedances add, parallel admittances add.
- $|Z|$ sets the amplitude ratio, $\arg Z$ the phase lead of voltage over current. Inductive circuits make current lag, capacitive ones make it lead.
- The series RLC circuit resonates at $\omega_0 = 1/\sqrt{LC}$ where $Z = R$; sharpness is $Q = \omega_0 L/R$, bandwidth $R/L$. Geometrically $Z(\omega)$ runs along a line and $Y(\omega)$ around a circle.
- A Runge–Kutta simulation of the circuit ODE converges to the phasor prediction once the transient decays — complex algebra really does solve the differential equation.

**Next:** notebook 02 explains *why* this works in general: it solves linear ODEs with the ansatz $e^{\lambda t}$, shows that complex roots $\lambda$ mean oscillation, and treats the damped and driven oscillator — the mechanical twin of the RLC circuit — including the transient we saw here.